# State Bench Long-Context: Hybrid vs Transformer

This notebook compares the matched 275M hybrid and transformer checkpoints on the `jacksonp-state-bench-lc` tasks. It reads per-example prediction artifacts, derives exact prompt-token length as `num_tokens_all - num_tokens`, and supports task-level and accuracy-versus-prompt-length analysis.

In [ ]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import aesthetics as aes  # noqa: E402

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)

## Configuration

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "results" / "jacksonp-state-bench-lc" / "all_results.json"
TASK_RE = re.compile(r"state_bench_lc_(.+)_(aperiodic|periodic|r_trivial)$")
COMPLEXITY_ORDER = ["aperiodic", "periodic", "r_trivial"]
MODEL_ORDER = ["Transformer", "Hybrid"]
MODEL_COLORS = {"Transformer": "#DD0426", "Hybrid": "#8304DE"}
# Even-width log10 bins are appropriate for prompts spanning 17 to ~131k tokens.
LENGTH_EDGES = np.logspace(1, np.log10(200_000), 16)
MIN_BIN_EXAMPLES = 30


def model_label(result: dict) -> str:
    text = f"{result.get('experiment_name', '')} {result.get('model', '')}".lower()
    return "Transformer" if "transformer" in text else "Hybrid"


def nested_metric(metrics: dict, metric: str) -> float:
    value = metrics
    for key in metric.split(":"):
        value = value[key]
    return float(value)


def artifact_path(result: dict, task: dict, kind: str) -> Path:
    root = RESULTS_PATH.parent / result["experiment_id"] / "main" / kind
    matches = sorted(root.glob(f"*/*_{task['task_hash'][-6:]}-{kind}.jsonl"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {kind} artifact for {task['task']}; found {matches}"
        )
    return matches[0]

## Load per-example results

The two runs evaluate the same examples. Prompt tokens come directly from each model output; all answer choices for an example share the same derived prompt length.

In [ ]:
raw_results = json.loads(RESULTS_PATH.read_text())
rows = []
for result in raw_results:
    model = model_label(result)
    for task in result["tasks"]:
        match = TASK_RE.fullmatch(task["task"])
        if match is None:
            continue
        domain, complexity = match.groups()
        metric = task["primary_metric"]
        request_stream = (
            artifact_path(result, task, "requests").open()
            if complexity == "r_trivial"
            else None
        )
        with artifact_path(result, task, "predictions").open() as stream:
            for line in stream:
                prediction = json.loads(line)
                request = (
                    json.loads(next(request_stream))
                    if request_stream is not None
                    else {}
                )
                doc = request.get("doc", {})
                if request and prediction["doc_id"] != request["doc_id"]:
                    raise ValueError(
                        f"Prediction/request mismatch: {task['task']} doc {prediction['doc_id']}"
                    )
                prompt_lengths = [
                    output["num_tokens_all"] - output["num_tokens"]
                    for output in prediction["model_output"]
                ]
                if len(set(prompt_lengths)) != 1:
                    raise ValueError(
                        f"Choice-dependent prompt length: {task['task']} doc {prediction['doc_id']}"
                    )
                rows.append(
                    {
                        "model": model,
                        "task": task["task"],
                        "domain": domain.replace("_", " ").title(),
                        "complexity": complexity.replace("_", "-"),
                        "doc_id": prediction["doc_id"],
                        "native_id": prediction["native_id"],
                        "prompt_tokens": prompt_lengths[0],
                        "score": nested_metric(prediction["instance_metrics"], metric),
                        "padding_lines": doc.get("num_filler_lines"),
                        "padding_complexity": doc.get("filler_distraction")
                        or ("none" if doc else None),
                    }
                )
        if request_stream is not None:
            request_stream.close()

item_df = pd.DataFrame(rows)
item_df["model"] = pd.Categorical(item_df["model"], MODEL_ORDER, ordered=True)
item_df["complexity"] = pd.Categorical(
    item_df["complexity"],
    [value.replace("_", "-") for value in COMPLEXITY_ORDER],
    ordered=True,
)
print(
    f"Loaded {len(item_df):,} example/model rows across {item_df['task'].nunique()} task variants"
)
display(
    item_df.groupby("model", observed=True).agg(
        examples=("score", "size"),
        accuracy=("score", "mean"),
        min_tokens=("prompt_tokens", "min"),
        max_tokens=("prompt_tokens", "max"),
    )
)

## Task-level performance

In [ ]:
task_df = item_df.groupby(
    ["domain", "complexity", "model"], observed=True, as_index=False
).agg(accuracy=("score", "mean"), n=("score", "size"))
task_table = task_df.pivot(
    index=["domain", "complexity"], columns="model", values="accuracy"
)
task_table["Hybrid - Transformer"] = task_table["Hybrid"] - task_table["Transformer"]
display(
    task_table.style.format("{:.1%}").background_gradient(
        subset=["Hybrid - Transformer"], cmap="PiYG", vmin=-0.12, vmax=0.12
    )
)

plot_task_df = task_df.copy()
plot_task_df["task label"] = (
    plot_task_df["domain"] + " · " + plot_task_df["complexity"].astype(str)
)
fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN, 4.2))
sns.barplot(
    data=plot_task_df,
    y="task label",
    x="accuracy",
    hue="model",
    hue_order=MODEL_ORDER,
    palette=MODEL_COLORS,
    ax=ax,
)
ax.set(
    xlabel="Accuracy", ylabel=None, xlim=(0, 1), title="State Bench LC accuracy by task"
)
ax.xaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.legend(title=None, frameon=False, loc="lower right")
sns.despine()
fig.tight_layout()
plt.show()

## Accuracy vs prompt length

Points use powers-of-two prompt-token bins; bands are Wilson 95% intervals. Edit `LENGTH_EDGES` or `MIN_BIN_EXAMPLES` above to change the resolution.

In [ ]:
def wilson_interval(successes: pd.Series, n: pd.Series) -> tuple[pd.Series, pd.Series]:
    z = 1.96
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    radius = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominator
    return center - radius, center + radius


item_df["length_bin"] = pd.cut(
    item_df["prompt_tokens"], LENGTH_EDGES, right=False, include_lowest=True
)
item_df["length_mid"] = (
    item_df["length_bin"]
    .map(
        lambda value: math.sqrt(value.left * value.right) if pd.notna(value) else np.nan
    )
    .astype(float)
)
length_df = item_df.groupby(
    ["domain", "complexity", "model", "length_bin", "length_mid"],
    observed=True,
    as_index=False,
).agg(accuracy=("score", "mean"), successes=("score", "sum"), n=("score", "size"))
length_df = length_df[length_df["n"] >= MIN_BIN_EXAMPLES].copy()
length_df["ci_low"], length_df["ci_high"] = wilson_interval(
    length_df["successes"], length_df["n"]
)

panels = (
    item_df[["domain", "complexity"]]
    .drop_duplicates()
    .sort_values(["domain", "complexity"])
)
fig, axes = plt.subplots(
    4, 3, figsize=(aes.PAPER_WIDTH_IN * 1.45, 8.0), sharex=True, sharey=True
)
for ax in axes.flat:
    ax.set_visible(False)
for ax, (_, panel) in zip(axes.flat, panels.iterrows()):
    ax.set_visible(True)
    subset = length_df[
        (length_df["domain"] == panel["domain"])
        & (length_df["complexity"] == panel["complexity"])
    ]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("length_mid")
        x = curve["length_mid"].to_numpy(float)
        y = curve["accuracy"].to_numpy(float)
        ax.plot(
            x, y, marker="o", ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    ax.set_xscale("log", base=10)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.set_title(f"{panel['domain']} · {panel['complexity']}", fontsize=8)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
fig.supxlabel("Prompt tokens (log₁₀ scale)")
fig.supylabel("Accuracy")
handles = [
    plt.Line2D([0], [0], color=MODEL_COLORS[m], marker="o", label=m)
    for m in MODEL_ORDER
]
fig.legend(handles=handles, loc="center right", frameon=False)
fig.suptitle("Accuracy vs prompt length by task", y=1.01)
fig.tight_layout(rect=(0, 0, 0.93, 1))
plt.show()

## Paired hybrid-minus-transformer differences by length

In [ ]:
paired = (
    item_df.pivot(
        index=[
            "domain",
            "complexity",
            "native_id",
            "prompt_tokens",
            "length_bin",
            "length_mid",
        ],
        columns="model",
        values="score",
    )
    .dropna()
    .reset_index()
)
paired["delta"] = paired["Hybrid"] - paired["Transformer"]
delta_df = paired.groupby(
    ["domain", "complexity", "length_bin", "length_mid"], observed=True, as_index=False
).agg(delta=("delta", "mean"), sd=("delta", "std"), n=("delta", "size"))
delta_df = delta_df[delta_df["n"] >= MIN_BIN_EXAMPLES].copy()
delta_df["se"] = delta_df["sd"] / np.sqrt(delta_df["n"])
delta_df["ci_low"] = delta_df["delta"] - 1.96 * delta_df["se"]
delta_df["ci_high"] = delta_df["delta"] + 1.96 * delta_df["se"]

fig, axes = plt.subplots(
    4, 3, figsize=(aes.PAPER_WIDTH_IN * 1.45, 8.0), sharex=True, sharey=True
)
for ax in axes.flat:
    ax.set_visible(False)
for ax, (_, panel) in zip(axes.flat, panels.iterrows()):
    ax.set_visible(True)
    curve = delta_df[
        (delta_df["domain"] == panel["domain"])
        & (delta_df["complexity"] == panel["complexity"])
    ].sort_values("length_mid")
    x = curve["length_mid"].to_numpy(float)
    y = curve["delta"].to_numpy(float)
    ax.plot(x, y, marker="o", ms=2.8, lw=1.2, color=MODEL_COLORS["Hybrid"])
    ax.fill_between(
        x,
        curve["ci_low"].to_numpy(float),
        curve["ci_high"].to_numpy(float),
        color=MODEL_COLORS["Hybrid"],
        alpha=0.14,
    )
    ax.axhline(0, color="#555555", lw=0.8)
    ax.set_xscale("log", base=10)
    ax.set_ylim(-0.35, 0.35)
    ax.grid(alpha=0.2)
    ax.set_title(f"{panel['domain']} · {panel['complexity']}", fontsize=8)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value * 100:.0f} pts")
fig.supxlabel("Prompt tokens (log₁₀ scale)")
fig.supylabel("Hybrid − transformer accuracy")
fig.suptitle("Paired architecture advantage vs prompt length", y=1.01)
fig.tight_layout()
plt.show()

## R-trivial breakout A: padding amount

For r-trivial Integer Code examples, padding amount is the dataset's `num_filler_lines`: 0, 18, 198, or 1,998 filler lines. The table and plot marginalize over padding complexity within each nonzero amount.

In [ ]:
rtrivial_df = item_df[item_df["complexity"] == "r-trivial"].copy()
rtrivial_df["padding_lines"] = rtrivial_df["padding_lines"].astype(int)
padding_amount_df = rtrivial_df.groupby(
    ["padding_lines", "model"], observed=True, as_index=False
).agg(
    accuracy=("score", "mean"),
    n=("score", "size"),
    median_prompt_tokens=("prompt_tokens", "median"),
)
padding_amount_table = padding_amount_df.pivot(
    index="padding_lines", columns="model", values="accuracy"
)
padding_amount_table["Hybrid - Transformer"] = (
    padding_amount_table["Hybrid"] - padding_amount_table["Transformer"]
)
display(
    padding_amount_table.style.format("{:.1%}").background_gradient(
        subset=["Hybrid - Transformer"], cmap="PiYG", vmin=-0.15, vmax=0.15
    )
)
display(
    padding_amount_df.pivot(
        index="padding_lines", columns="model", values=["n", "median_prompt_tokens"]
    )
)

fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN * 0.75, 2.8))
for model in MODEL_ORDER:
    curve = padding_amount_df[padding_amount_df["model"] == model].sort_values(
        "padding_lines"
    )
    ax.plot(
        curve["padding_lines"].astype(str),
        curve["accuracy"],
        marker="o",
        lw=1.5,
        color=MODEL_COLORS[model],
        label=model,
    )
ax.set(
    xlabel="Padding amount (filler lines)",
    ylabel="Accuracy",
    ylim=(0, 1),
    title="R-trivial accuracy by padding amount",
)
ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, title=None)
sns.despine()
fig.tight_layout()
plt.show()

## R-trivial breakout B: padding complexity

Padding complexity is the dataset's `filler_distraction` level (`easy`, `medium`, or `hard`). Unpadded controls are shown as `none`; nonzero-complexity results marginalize over the three padding amounts.

In [ ]:
PADDING_COMPLEXITY_ORDER = ["none", "easy", "medium", "hard"]
rtrivial_df["padding_complexity"] = pd.Categorical(
    rtrivial_df["padding_complexity"], PADDING_COMPLEXITY_ORDER, ordered=True
)
padding_complexity_df = rtrivial_df.groupby(
    ["padding_complexity", "model"], observed=True, as_index=False
).agg(
    accuracy=("score", "mean"),
    n=("score", "size"),
    median_prompt_tokens=("prompt_tokens", "median"),
)
padding_complexity_table = padding_complexity_df.pivot(
    index="padding_complexity", columns="model", values="accuracy"
)
padding_complexity_table["Hybrid - Transformer"] = (
    padding_complexity_table["Hybrid"] - padding_complexity_table["Transformer"]
)
display(
    padding_complexity_table.style.format("{:.1%}").background_gradient(
        subset=["Hybrid - Transformer"], cmap="PiYG", vmin=-0.15, vmax=0.15
    )
)
display(
    padding_complexity_df.pivot(
        index="padding_complexity",
        columns="model",
        values=["n", "median_prompt_tokens"],
    )
)

fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN * 0.75, 2.8))
sns.barplot(
    data=padding_complexity_df,
    x="padding_complexity",
    y="accuracy",
    hue="model",
    hue_order=MODEL_ORDER,
    order=PADDING_COMPLEXITY_ORDER,
    palette=MODEL_COLORS,
    ax=ax,
)
ax.set(
    xlabel="Padding complexity",
    ylabel="Accuracy",
    ylim=(0, 1),
    title="R-trivial accuracy by padding complexity",
)
ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, title=None)
sns.despine()
fig.tight_layout()
plt.show()

## Log-length trend models

For each task, a binomial GLM estimates `accuracy ~ model * centered log10(prompt tokens)`. The hybrid coefficient is its log-odds advantage at that task's median length; the interaction is the change in hybrid advantage for each 10× increase in prompt length. These linear log-length trends are summaries of the richer curves above.

In [ ]:
trend_rows = []
for (domain, complexity), group in item_df.groupby(
    ["domain", "complexity"], observed=True
):
    group = group.copy()
    group["log10_tokens_centered"] = np.log10(group["prompt_tokens"]) - np.log10(
        group["prompt_tokens"].median()
    )
    fit = smf.glm(
        "score ~ C(model, Treatment(reference='Transformer')) * log10_tokens_centered",
        data=group,
        family=sm.families.Binomial(),
    ).fit()
    model_term = "C(model, Treatment(reference='Transformer'))[T.Hybrid]"
    interaction = model_term + ":log10_tokens_centered"
    trend_rows.append(
        {
            "domain": domain,
            "complexity": complexity,
            "median_tokens": group["prompt_tokens"].median(),
            "hybrid_odds_ratio_at_median": np.exp(fit.params[model_term]),
            "hybrid_at_median_p": fit.pvalues[model_term],
            "hybrid_x_10x_length_log_odds": fit.params[interaction],
            "length_interaction_p": fit.pvalues[interaction],
        }
    )
trend_df = pd.DataFrame(trend_rows).sort_values(["domain", "complexity"])
display(
    trend_df.style.format(
        {
            "median_tokens": "{:,.0f}",
            "hybrid_odds_ratio_at_median": "{:.2f}",
            "hybrid_at_median_p": "{:.2g}",
            "hybrid_x_10x_length_log_odds": "{:+.3f}",
            "length_interaction_p": "{:.2g}",
        }
    )
)